In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "easyocr", "opencv-python", "pandas", "openpyxl",
                       "pillow", "tqdm", "gradio", "numpy", "-q"])
print("All installed!")

All installed!


In [2]:
import os

DATA_DIR = "data/raw"
images = [f for f in os.listdir(DATA_DIR)
          if f.lower().endswith((".jpg", ".jpeg", ".png"))]

print(f"Found {len(images)} images")
print("\nFirst 5:")
for img in images[:5]:
    print(f"  {img}")

Found 30 images

First 5:
  1.png
  10.jpg
  11.png
  12.png
  13.png


In [11]:
import os, re
import cv2
import numpy as np
import pandas as pd
import easyocr
from PIL import Image
from tqdm import tqdm

DATA_DIR    = "data/raw"
OUTPUT_DIR  = "output/tables"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading EasyOCR...")
reader = easyocr.Reader(["en"], gpu=False)
print("Ready!")

Using CPU. Note: This module is much faster with a GPU.


Loading EasyOCR...
Ready!


In [ ]:
def preprocess(img_path):
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    if w < 1200:
        scale = 1200 / w
        img   = cv2.resize(img, (1200, int(h * scale)),
                           interpolation=cv2.INTER_CUBIC)
    gray    = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray    = cv2.GaussianBlur(gray, (3, 3), 0)
    binary  = cv2.adaptiveThreshold(gray, 255,
                                    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY_INV, 15, 8)
    return img, gray, binary

def detect_table(img, binary):
    kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    dilated = cv2.dilate(binary, kernel, iterations=3)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    h, w = img.shape[:2]
    best, best_area = None, 0
    for cnt in contours:
        x, y, cw, ch = cv2.boundingRect(cnt)
        area = cw * ch
        if area > 0.10 * h * w and area > best_area:
            best      = (x, y, cw, ch)
            best_area = area
    return best

def has_borders(gray_crop):
  
    h, w = gray_crop.shape
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (w // 4, 1))
    h_lines  = cv2.morphologyEx(~gray_crop, cv2.MORPH_OPEN, h_kernel)
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, h // 4))
    v_lines  = cv2.morphologyEx(~gray_crop, cv2.MORPH_OPEN, v_kernel)
    h_score  = np.sum(h_lines) / (255 * h * w)
    v_score  = np.sum(v_lines) / (255 * h * w)
    return (h_score + v_score) > 0.02  

def detect_grid_bordered(gray_crop):
    
    h, w = gray_crop.shape

    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (w // 4, 1))
    h_lines  = cv2.morphologyEx(~gray_crop, cv2.MORPH_OPEN, h_kernel)
    h_proj   = np.sum(h_lines, axis=1) / 255

    row_ys = [0]
    for i in range(1, h - 1):
        if h_proj[i] > w * 0.3:
            if i - row_ys[-1] > 10:
                row_ys.append(i)
    if row_ys[-1] < h - 10:
        row_ys.append(h)

    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, h // 4))
    v_lines  = cv2.morphologyEx(~gray_crop, cv2.MORPH_OPEN, v_kernel)
    v_proj   = np.sum(v_lines, axis=0) / 255

    col_xs = [0]
    for i in range(1, w - 1):
        if v_proj[i] > h * 0.3:
            if i - col_xs[-1] > 10:
                col_xs.append(i)
    if col_xs[-1] < w - 10:
        col_xs.append(w)

    return row_ys, col_xs

def detect_grid_borderless(gray_crop):
    h, w = gray_crop.shape

    row_proj = np.sum(~gray_crop, axis=1) / 255
    smoothed = np.convolve(row_proj, np.ones(3)/3, mode='same')

    row_ys   = [0]
    in_text  = False
    for i in range(h):
        if smoothed[i] > 5 and not in_text:
            if i - row_ys[-1] > 5:
                row_ys.append(max(0, i - 2))
            in_text = True
        elif smoothed[i] <= 2 and in_text:
            row_ys.append(min(h, i + 2))
            in_text = False
    if row_ys[-1] < h - 5:
        row_ys.append(h)

    row_ys = sorted(set(row_ys))

    col_proj  = np.sum(~gray_crop, axis=0) / 255
    smoothed_c = np.convolve(col_proj, np.ones(5)/5, mode='same')

    col_xs   = [0]
    in_text  = False
    for i in range(w):
        if smoothed_c[i] > 2 and not in_text:
            if i - col_xs[-1] > 15:
                col_xs.append(max(0, i - 5))
            in_text = True
        elif smoothed_c[i] <= 1 and in_text:
            col_xs.append(min(w, i + 5))
            in_text = False
    if col_xs[-1] < w - 5:
        col_xs.append(w)

    col_xs = sorted(set(col_xs))

    if len(row_ys) < 3:
        step   = h // 8
        row_ys = list(range(0, h, step)) + [h]
    if len(col_xs) < 3:
        step   = w // 5
        col_xs = list(range(0, w, step)) + [w]

    return row_ys, col_xs

def ocr_row(gray_crop, y1, y2):
    row_img = gray_crop[y1:y2, :]
    if row_img.shape[0] < 5:
        return []
    scale   = max(1, 60 // row_img.shape[0])
    row_img = cv2.resize(row_img,
                         (row_img.shape[1] * scale, row_img.shape[0] * scale),
                         interpolation=cv2.INTER_CUBIC)
    row_img = cv2.copyMakeBorder(row_img, 10, 10, 10, 10,
                                 cv2.BORDER_CONSTANT, value=255)
    results = reader.readtext(row_img, detail=1, paragraph=False)
    if not results:
        return []
    detections = []
    for bbox, text, conf in results:
        xs       = [pt[0] for pt in bbox]
        x_center = sum(xs) / len(xs)
        detections.append((x_center, text.strip()))
    detections.sort(key=lambda x: x[0])
    return [d[1] for d in detections]

def extract_cells_bordered(gray_crop, row_ys, col_xs):
    """Cell-by-cell extraction for bordered tables"""
    rows_data = []
    for r in range(len(row_ys) - 1):
        y1, y2 = row_ys[r], row_ys[r + 1]
        if y2 - y1 < 8:
            continue
        row = []
        for c in range(len(col_xs) - 1):
            x1, x2 = col_xs[c], col_xs[c + 1]
            if x2 - x1 < 8:
                continue
            cell   = gray_crop[y1:y2, x1:x2]
            ch, cw = cell.shape
            scale  = max(1, 60 // ch, 100 // cw)
            if scale > 1:
                cell = cv2.resize(cell, (cw * scale, ch * scale),
                                  interpolation=cv2.INTER_CUBIC)
            kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
            cell   = cv2.filter2D(cell, -1, kernel)
            _, cell = cv2.threshold(cell, 0, 255,
                                    cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            cell   = cv2.copyMakeBorder(cell, 10, 10, 10, 10,
                                        cv2.BORDER_CONSTANT, value=255)
            result = reader.readtext(cell, detail=0, paragraph=True)
            text   = " ".join(result).strip()
            row.append(text)
        if any(t.strip() for t in row):
            rows_data.append(row)
    return rows_data

def extract_rows_borderless(gray_crop, row_ys):
    
    rows_data = []
    for r in range(len(row_ys) - 1):
        y1, y2 = row_ys[r], row_ys[r + 1]
        if y2 - y1 < 8:
            continue
        row = ocr_row(gray_crop, y1, y2)
        if row:
            rows_data.append(row)
    return rows_data

def save_table(rows_data, out_path):
    if not rows_data:
        return None
    max_cols = max(len(r) for r in rows_data)
    for r in rows_data:
        while len(r) < max_cols:
            r.append("")
    df = pd.DataFrame(rows_data[1:], columns=rows_data[0])
    df.to_excel(out_path, index=False)
    return df

print("Functions reloaded!")

Functions reloaded!


In [14]:
test_img = "data/raw/6.png"
img, gray, binary = preprocess(test_img)
bbox              = detect_table(img, binary)

if bbox:
    x, y, w, h = bbox
    gray_crop  = gray[y:y+h, x:x+w]
else:
    gray_crop = gray

row_ys    = detect_rows(gray_crop)
rows_data = extract_rows(gray_crop, row_ys)

print(f"Rows detected: {len(row_ys)-1}")
print(f"Data rows extracted: {len(rows_data)}")
for i, row in enumerate(rows_data[:4]):
    print(f"Row {i}: {row}")

Rows detected: 7
Data rows extracted: 2
Row 0: ['Name', 'Score']
Row 1: ['Annie', 'Annie', 'Annie', 'Beth', 'Beth', 'Beth', 'Dana', 'Cathy', 'Cathy', '44', '33', '88', '77', '99', '11', '55', '66', '22']


In [15]:
from PIL import Image
img_check = Image.open("data/raw/2.png")
print(f"Size: {img_check.size}")
print(f"Mode: {img_check.mode}")

Size: (435, 569)
Mode: P


In [13]:
SUPPORTED   = (".jpg", ".jpeg", ".png")
image_files = sorted([f for f in os.listdir(DATA_DIR)
                      if f.lower().endswith(SUPPORTED)])

results = []
print(f"Processing {len(image_files)} images...\n")

for fname in tqdm(image_files):
    img_path = os.path.join(DATA_DIR, fname)
    stem     = os.path.splitext(fname)[0]
    out_path = os.path.join(OUTPUT_DIR, f"{stem}_table.xlsx")

    try:
        img, gray, binary = preprocess(img_path)
        bbox              = detect_table(img, binary)

        if bbox:
            x, y, w, h = bbox
            gray_crop  = gray[y:y+h, x:x+w]
        else:
            gray_crop  = gray

        if has_borders(gray_crop):
            row_ys, col_xs = detect_grid_bordered(gray_crop)
            rows_data      = extract_cells_bordered(gray_crop, row_ys, col_xs)
            table_type     = "bordered"
        else:
            row_ys, _  = detect_grid_borderless(gray_crop)
            rows_data  = extract_rows_borderless(gray_crop, row_ys)
            table_type = "borderless"

        df   = save_table(rows_data, out_path)
        rows = len(df) if df is not None else 0
        cols = len(df.columns) if df is not None else 0

        results.append({
            "file":        fname,
            "table_type":  table_type,
            "table_found": bbox is not None,
            "rows":        rows,
            "cols":        cols,
            "cells":       rows * cols,
            "status":      "success"
        })

    except Exception as e:
        print(f"Error on {fname}: {e}")
        results.append({
            "file": fname, "table_type": "unknown",
            "table_found": False, "rows": 0,
            "cols": 0, "cells": 0,
            "status": f"error: {e}"
        })

results_df = pd.DataFrame(results)
results_df.to_excel("output/pipeline_summary.xlsx", index=False)
print("\nDone!")
print(f"\nBordered tables  : {(results_df['table_type']=='bordered').sum()}")
print(f"Borderless tables: {(results_df['table_type']=='borderless').sum()}")

Processing 30 images...



  3%|▎         | 1/30 [00:04<02:18,  4.79s/it]c:\Users\meera\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
 23%|██▎       | 7/30 [01:00<02:25,  6.34s/it]c:\Users\meera\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
 53%|█████▎    | 16/30 [02:24<01:30,  6.47s/it]c:\Users\meera\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
 63%|██████▎   | 19/30 [03:12<01:53, 10.29s/it]c:\Users\meera\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no 


Done!

Bordered tables  : 20
Borderless tables: 10


In [16]:
import numpy as np

total        = len(results_df)
found        = results_df["table_found"].sum()
success      = (results_df["status"] == "success").sum()
avg_rows     = results_df[results_df["rows"] > 0]["rows"].mean()
avg_cols     = results_df[results_df["cols"] > 0]["cols"].mean()
avg_cells    = results_df[results_df["cells"] > 0]["cells"].mean()
detection_rt = found / total * 100
success_rt   = success / total * 100

print("\n" + "="*55)
print("  TABLE EXTRACTION METRICS")
print("="*55)
print(f"  Images processed       : {total}")
print(f"  Table detected         : {found}/{total} → {detection_rt:.1f}%")
print(f"  Successfully extracted : {success}/{total} → {success_rt:.1f}%")
print(f"  Avg rows per table     : {avg_rows:.1f}")
print(f"  Avg cols per table     : {avg_cols:.1f}")
print(f"  Avg cells per table    : {avg_cells:.1f}")
print(f"  Excel files saved to   : output/tables/")

print(f"""
RESUME BULLET:
──────────────────────────────────────────────────────
Built a table extraction pipeline on {total} scanned
document images using OpenCV + EasyOCR; achieved
{detection_rt:.0f}% table detection rate and {success_rt:.0f}% successful
extraction rate; avg {avg_rows:.0f} rows x {avg_cols:.0f} cols reconstructed
per table, exported to structured Excel files.
──────────────────────────────────────────────────────
""")


  TABLE EXTRACTION METRICS
  Images processed       : 30
  Table detected         : 25/30 → 83.3%
  Successfully extracted : 30/30 → 100.0%
  Avg rows per table     : 10.2
  Avg cols per table     : 5.4
  Avg cells per table    : 60.9
  Excel files saved to   : output/tables/

RESUME BULLET:
──────────────────────────────────────────────────────
Built a table extraction pipeline on 30 scanned
document images using OpenCV + EasyOCR; achieved
83% table detection rate and 100% successful
extraction rate; avg 10 rows x 5 cols reconstructed
per table, exported to structured Excel files.
──────────────────────────────────────────────────────

